<div align='center'>

<img src='https://upload.wikimedia.org/wikipedia/commons/2/27/MnistExamples.png' width='580'/>

<br/>

# MNIST Handwritten Digit Classification
### A Deep Learning Approach Using Feedforward Neural Networks

---

| | |
|:---|:---|
| **Series** | *Deep Learning From Scratch* — Episode 01 |
| **Framework** | TensorFlow 2.x / Keras |
| **Dataset** | MNIST — Modified National Institute of Standards and Technology |
| **Architecture** | Fully Connected Neural Network (FCNN / MLP) |
| **Task** | Multi-class Image Classification (10 classes) |
| **Test Accuracy** | ≈ 97.06 % |
| **Author** | Prem Patel |

> *Deep Learning From Scratch* is a fully documented, project-based series where every concept — from raw data ingestion to model deployment — is explained at both an intuitive and rigorous mathematical level. Built for students, practitioners, and future researchers who want to truly understand what happens beneath the surface of modern deep learning systems.

</div>

---

## Abstract

This notebook presents a complete, end-to-end implementation of a **Feedforward Neural Network (FNN)** for classifying handwritten digit images drawn from the canonical **MNIST benchmark dataset**. Starting from raw uint8 pixel arrays, we traverse the full machine learning pipeline: exploratory data analysis, input normalization, architecture design, gradient-based optimization, training dynamics, held-out evaluation, probabilistic inference, and confusion analysis.

The model achieves **≈ 97% accuracy** on the held-out test partition using a shallow two-hidden-layer network — demonstrating that even parameter-efficient architectures can learn robust visual representations when given clean, well-normalized data.

All design decisions are explained from first principles. Mathematical formulations accompany every major operation to build a rigorous bridge between implementation and theory.

---

## Table of Contents

1. [Introduction & Problem Formulation](#introduction)
2. [Environment Setup & Dependencies](#setup)
3. [Dataset Acquisition & Structural Analysis](#dataset)
4. [Exploratory Data Analysis (EDA)](#eda)
5. [Data Preprocessing & Normalization](#preprocessing)
6. [Neural Network Architecture Design](#architecture)
7. [Model Compilation & Optimization Strategy](#compilation)
8. [Model Training & Learning Dynamics](#training)
9. [Evaluation on Unseen Test Data](#evaluation)
10. [Probabilistic Inference & Prediction Decoding](#inference)
11. [Confusion Matrix Analysis](#confusion)
12. [End-to-End Predictive System](#predictive)
13. [Conclusion, Limitations & Future Directions](#conclusion)
14. [References](#references)

---

<a id='introduction'></a>

## 1. Introduction & Problem Formulation

### 1.1 The MNIST Dataset

The **MNIST database** (*Modified National Institute of Standards and Technology*) is the most studied benchmark in the history of machine learning. Originally constructed by **Yann LeCun, Corinna Cortes, and Christopher Burges** from remixed NIST datasets, it contains:

| Property | Value |
|:---|:---|
| Total images | 70,000 |
| Training split | 60,000 images |
| Test split | 10,000 images |
| Image resolution | 28 × 28 pixels |
| Color space | Grayscale (single channel) |
| Pixel range | Integer ∈ [0, 255] |
| Classes | 10 (digits 0 – 9) |

MNIST is often called the *"Hello World" of deep learning* — simple enough to prototype quickly, yet structured enough to validate architectural decisions and training pipelines.

### 1.2 Why Neural Networks for Vision?

Classical machine learning methods (SVM, k-NN, decision trees) treat images as **flat feature vectors** and lose all spatial structure. Neural networks with non-linear activation functions learn **hierarchical representations**:

```
  Raw Pixels → Edges → Curves → Digit Parts → Complete Digit
```

Even without convolutions, a sufficiently deep **Fully Connected Neural Network (FCNN)** can learn discriminative features. This episode intentionally uses an FCNN to establish a clear baseline — the limitations of this approach motivate the introduction of CNNs in Episode 02.

### 1.3 Formal Problem Statement

We frame this as a **supervised multi-class classification** problem. Given a dataset of $N$ labeled image-label pairs:

$$
\mathcal{D} = \{(\mathbf{x}^{(i)}, y^{(i)})\}_{i=1}^{N}, \quad \mathbf{x}^{(i)} \in \mathbb{R}^{784}, \quad y^{(i)} \in \{0, 1, \ldots, 9\}
$$

our goal is to learn a mapping $f_{\boldsymbol{\theta}}$ parameterized by weights $\boldsymbol{\theta}$:

$$
f_{\boldsymbol{\theta}}: \mathbb{R}^{784} \longrightarrow \Delta^{9}
$$

where $\Delta^9$ is the 9-dimensional probability simplex (outputs sum to 1), such that the predicted class $\hat{y} = \arg\max_k f_{\boldsymbol{\theta}}(\mathbf{x})_k$ minimizes classification error on unseen test data.

>  **Note for Future Researchers:** The $784$-dimensional input arises from flattening the $28 \times 28$ pixel matrix, which **destroys spatial locality**. This is the central motivation behind Convolutional Neural Networks (CNNs), which exploit translation equivariance — covered in Episode 02.

---

<a id='setup'></a>

## 2. Environment Setup & Dependencies

We import all necessary libraries before any computation. Each serves a well-defined role:

| Library | Role in Pipeline |
|:---|:---|
| `numpy` | N-dimensional array operations, linear algebra, argmax decoding |
| `matplotlib` | Image visualization, training curve plotting |
| `seaborn` | Statistical heatmaps (confusion matrix) |
| `opencv (cv2)` | Image I/O and resizing for custom inference |
| `PIL (Pillow)` | High-level image manipulation utilities |
| `tensorflow` | Automatic differentiation, GPU acceleration |
| `keras` | High-level neural network API (built on TensorFlow) |

**Reproducibility:** We fix TensorFlow's global random seed before any model instantiation. This ensures deterministic weight initialization and consistent results across independent runs — a non-negotiable practice in research.

$$
\texttt{tf.random.set\_seed}(s) \implies \text{all stochastic ops in the graph use seed } s
$$

In [ ]:
# 
# § 2.1 — Scientific Computing & Visualization Stack
# 
import numpy as np                          # Numerical arrays & linear algebra
import matplotlib.pyplot as plt             # 2D plotting and image rendering
import seaborn as sns                       # Statistical visualization

# 
# § 2.2 — Image Processing
# 
import cv2                                  # OpenCV: image I/O & spatial transforms
from google.colab.patches import cv2_imshow # Colab-compatible image display patch
from PIL import Image                       # PIL: high-level image manipulation

# 
# § 2.3 — Deep Learning Framework (TensorFlow / Keras)
# 
import tensorflow as tf
tf.random.set_seed(3)                       # Fix global random seed → reproducibility

from tensorflow import keras
from tensorflow.keras.datasets import mnist
from tensorflow.math import confusion_matrix

# Version Report 
print(f"TensorFlow version  : {tf.__version__}")
print(f"NumPy version       : {np.__version__}")
print("All dependencies loaded successfully ")

---
<a id='dataset'></a>

## 3. Dataset Acquisition & Structural Analysis

### 3.1 Loading MNIST via Keras

Keras ships with a built-in MNIST loader that fetches the dataset from Google's storage on first call, caches it locally at `~/.keras/datasets/mnist.npz`, and returns two tuples of pre-split NumPy arrays:

```python
(X_train, y_train), (X_test, y_test) = mnist.load_data()
```

| Variable | Shape | Dtype | Description |
|:---|:---|:---|:---|
| `X_train` | `(60000, 28, 28)` | `uint8` | Training image tensors |
| `y_train` | `(60000,)` | `uint8` | Training class labels ∈ {0,...,9} |
| `X_test` | `(10000, 28, 28)` | `uint8` | Test image tensors |
| `y_test` | `(10000,)` | `uint8` | Test class labels ∈ {0,...,9} |

>  **Data Leakage Warning:** The test set must remain **completely untouched** throughout the development cycle — including during architecture selection, hyperparameter tuning, and preprocessing decisions. Evaluating on test data before finalizing the model inflates reported performance and invalidates the experimental protocol. This is one of the most common methodological errors in applied ML.

In [ ]:
# Load MNIST — downloads on first run (~11 MB), cached on subsequent runs
(X_train, y_train), (X_test, y_test) = mnist.load_data()
print("MNIST dataset loaded ")

In [ ]:
# Confirm underlying data structure and element type
print(f"Type of X_train        : {type(X_train)}")
print(f"Element dtype (X_train): {X_train.dtype}  → unsigned 8-bit integer ∈ [0, 255]")
print(f"Element dtype (y_train): {y_train.dtype}  → unsigned 8-bit integer ∈ {{0,...,9}}")

In [ ]:
# Inspect tensor dimensions across all four arrays
print("" * 55)
print(f"  X_train : {X_train.shape}   → (N_train, H, W)")
print(f"  y_train : {y_train.shape}        → (N_train,)")
print(f"  X_test  : {X_test.shape}    → (N_test,  H, W)")
print(f"  y_test  : {y_test.shape}        → (N_test,)")
print("" * 55)
print(f"  Total images  : {X_train.shape[0] + X_test.shape[0]:,}")
print(f"  Image size    : {X_train.shape[1]} × {X_train.shape[2]} pixels")
print(f"  Pixel range   : [{X_train.min()}, {X_train.max()}]")
print(f"  Memory (train): {X_train.nbytes / 1e6:.1f} MB")

### 3.2 The Raw Pixel Representation

Each image is a **rank-2 integer tensor** of shape $(28, 28)$. Every scalar entry is a **grayscale luminance value** $x_{ij} \in \mathbb{Z}_{[0,255]}$:

$$
\mathbf{X}^{(n)} \in \mathbb{Z}_{[0,255]}^{28 \times 28}, \qquad
x_{ij}^{(n)} = \begin{cases} 0 & \text{pure black (background)} \\ 255 & \text{pure white (ink stroke)} \\ \text{intermediate} & \text{anti-aliased edge pixels} \end{cases}
$$

Printing the raw matrix of a sample image lets us build concrete intuition for what the model actually receives. The vast majority of entries are **zero** (background), with non-zero clusters forming the digit stroke.

In [ ]:
# Print the raw pixel matrix for a single training image
# Observe: sparse non-zero regions correspond to ink strokes
idx = 10
print(f"Raw pixel matrix — Training image [{idx}]  (True Label: {y_train[idx]})")
print("" * 60)
print(X_train[idx])

In [ ]:
# Confirm single image dimensions
print(f"Shape of X_train[10]    : {X_train[10].shape}")
print(f"Total pixels per image  : {X_train[10].size}  (= 28 × 28)")
print(f"After flattening for NN : ({X_train[10].size},) — 784-dim input vector")

---
<a id='eda'></a>

## 4. Exploratory Data Analysis (EDA)

### 4.1 Sample Image Visualization

Visual inspection of raw data is a mandatory first step in any rigorous ML project. It:

- Confirms correct data loading and expected value ranges
- Reveals image quality, alignment, and class variation
- Identifies potential issues (corrupted samples, label noise) before training

We display each sample under **two colormaps** to build dual intuition:
- `viridis` (default) → highlights intensity gradients; low values appear purple, high appear yellow
- `gray` → closest to real-world appearance; white strokes on black background

In [ ]:
# 
# § 4.1 — Visualize a training sample under two colormaps
# 
sample_idx = 29

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# viridis colormap
axes[0].imshow(X_train[sample_idx])
axes[0].set_title(f'viridis colormap\nLabel: {y_train[sample_idx]}', fontsize=12)
axes[0].axis('off')

# grayscale colormap (realistic rendering)
axes[1].imshow(X_train[sample_idx], cmap='gray')
axes[1].set_title(f'grayscale colormap\nLabel: {y_train[sample_idx]}', fontsize=12)
axes[1].axis('off')

plt.suptitle(f'MNIST Training Sample — Index {sample_idx}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 
# § 4.2 — Display one representative sample from each digit class (0–9)
# This reveals inter-class visual diversity
# 
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes = axes.flatten()

for digit in range(10):
    # Find first occurrence of this digit in y_train
    idx = np.where(y_train == digit)[0][0]
    axes[digit].imshow(X_train[idx], cmap='gray')
    axes[digit].set_title(f'Digit: {digit}', fontsize=11, fontweight='bold')
    axes[digit].axis('off')

plt.suptitle('One Representative Sample Per Class (0–9)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4.2 Label Encoding & Class Distribution

MNIST labels are stored as **integer scalars** $y \in \{0, 1, \ldots, 9\}$ — no explicit one-hot encoding. The `sparse_categorical_crossentropy` loss accepts this format directly.

**One-Hot vs. Sparse Label Comparison:**

| Format | Representation of class 3 | Loss function | Memory |
|:---|:---|:---|:---|
| **Integer (sparse)** | `3` | `sparse_categorical_crossentropy` | $O(1)$ per sample |
| **One-hot** | `[0,0,0,1,0,0,0,0,0,0]` | `categorical_crossentropy` | $O(K)$ per sample |

Both are mathematically equivalent. The sparse approach is preferred when $K$ (number of classes) is large.

In [ ]:
# Unique class labels present in each split
print(f"Classes in y_train : {np.unique(y_train)}")
print(f"Classes in y_test  : {np.unique(y_test)}")
print()

# Per-class sample count with ASCII bar chart
print("Class Distribution — Training Set:")
print("" * 45)
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique, counts):
    bar = '' * (cnt // 500)
    print(f"  Digit {cls} : {cnt:5,d} samples  {bar}")
print()
print(f"  Min class count : {counts.min():,d} | Max class count : {counts.max():,d}")
print(f"  Class imbalance ratio : {counts.max()/counts.min():.3f}  (near-balanced )")

---
<a id='preprocessing'></a>

## 5. Data Preprocessing & Normalization

### 5.1 The Case for Normalization

Raw MNIST pixel values span $[0, 255]$. Feeding unnormalized inputs to a neural network causes several pathologies:

**① Gradient instability.** Large input magnitudes produce large pre-activation values $z = \mathbf{w}^\top \mathbf{x} + b$. In the case of sigmoid/tanh activations, large $|z|$ drives outputs into saturation regions where $\sigma'(z) \approx 0$, causing **vanishing gradients** in the backward pass.

**② Poorly conditioned loss surface.** When input features have disparate scales, the loss landscape becomes elongated (high condition number), and gradient descent oscillates in narrow valleys instead of converging directly. Normalization isotropizes the curvature.

**③ Weight initialization mismatch.** Standard initializers (Xavier/Glorot, He) assume inputs have approximately zero mean and unit variance. Unnormalized inputs break these assumptions.

### 5.2 Min-Max Normalization to $[0, 1]$

We apply **min-max scaling** with global extrema $x_{\min} = 0$ and $x_{\max} = 255$:

$$
\tilde{\mathbf{X}} = \frac{\mathbf{X} - x_{\min}}{x_{\max} - x_{\min}} = \frac{\mathbf{X}}{255}
$$

This maps every pixel value to a real number in $[0.0, 1.0]$ while **preserving relative intensities**.

>  **Alternative — Z-Score Normalization:**
> $\tilde{\mathbf{X}} = (\mathbf{X} - \mu_{\text{train}}) / \sigma_{\text{train}}$ centers the distribution at zero with unit standard deviation. Common in pipelines that include Batch Normalization layers. For MNIST, both approaches yield comparable results.

In [ ]:
# 
# § 5.1 — Min-Max Normalization: map pixel values from [0,255] to [0,1]
# 

# Note: dividing uint8 by 255.0 automatically promotes to float64
X_train_scaled = X_train / 255.0
X_test_scaled  = X_test  / 255.0

print("Before normalization:")
print(f"  dtype     : {X_train.dtype}")
print(f"  min value : {X_train.min()}")
print(f"  max value : {X_train.max()}")
print()
print("After normalization:")
print(f"  dtype     : {X_train_scaled.dtype}")
print(f"  min value : {X_train_scaled.min():.4f}")
print(f"  max value : {X_train_scaled.max():.4f}")
print()
print("Normalization complete ")

In [ ]:
# Inspect the normalized pixel values of sample image [29]
# All values should now lie in [0.0, 1.0]
np.set_printoptions(precision=2, suppress=True, linewidth=100)
print(f"Normalized pixel matrix — Training image [29]  (Label: {y_train[29]}):")
print("" * 70)
print(X_train_scaled[29])

---
<a id='architecture'></a>

## 6. Neural Network Architecture Design

### 6.1 Architecture Topology

We construct a **3-layer Fully Connected Feedforward Network** (Multi-Layer Perceptron):

```
         
   INPUT               HIDDEN LAYER 1       HIDDEN LAYER 2       OUTPUT LAYER    
                                                                                 
  Flatten           Dense(50)         Dense(50)         Dense(10)       
  28×28 → 784          ReLU                 ReLU                 Sigmoid         
         
  784 neurons             784→50 W + 50 b        50→50 W + 50 b         50→10 W + 10 b
```

**Total Trainable Parameters:**

$$
\underbrace{(784 \times 50 + 50)}_{\text{Layer 1} = 39{,}250} \;+\; \underbrace{(50 \times 50 + 50)}_{\text{Layer 2} = 2{,}550} \;+\; \underbrace{(50 \times 10 + 10)}_{\text{Output} = 510} \;=\; \mathbf{42{,}810}
$$

### 6.2 Layer-by-Layer Mathematical Description

#### Layer 0 — `Flatten`: Vectorization

Reshapes the input matrix into a 1D vector without modifying values:
$$
\mathbf{x} = \text{vec}(\mathbf{X}) \in \mathbb{R}^{784}, \qquad \mathbf{X} \in \mathbb{R}^{28 \times 28}
$$

#### Layers 1 & 2 — `Dense(50, relu)`: Affine + Non-linear Transform

Each hidden layer applies a learned **affine transformation** followed by an **element-wise non-linearity**:
$$
\mathbf{h}^{(l)} = \text{ReLU}\!\left(\mathbf{W}^{(l)} \mathbf{h}^{(l-1)} + \mathbf{b}^{(l)}\right)
$$

**ReLU (Rectified Linear Unit):**
$$
\text{ReLU}(z) = \max(0, z), \qquad \frac{\partial\, \text{ReLU}(z)}{\partial z} = \mathbf{1}[z > 0]
$$

ReLU introduces non-linearity while maintaining a **gradient of 1** for positive activations — directly countering the vanishing gradient problem that plagues sigmoid/tanh in deep networks.

#### Layer 3 — `Dense(10, sigmoid)`: Output Scoring

The output layer produces 10 independent scores, one per digit class. **Sigmoid** maps each score to $(0, 1)$:
$$
\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad \sigma'(z) = \sigma(z)(1 - \sigma(z))
$$

>  **Sigmoid vs. Softmax — Research Note:**
> For multi-class classification, **Softmax** is theoretically more appropriate as it produces a proper probability distribution over all classes:
> $$\text{softmax}(\mathbf{z})_k = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$
> Sigmoid applied independently to each output does **not** guarantee outputs sum to 1. In practice on MNIST, `argmax` decoding renders this difference negligible. We will use Softmax in more rigorous experiments starting from Episode 02.

In [ ]:
# 
# § 6.1 — Define the Sequential Feedforward Neural Network
# 
model = keras.Sequential([

    #  Input Preprocessing 
    # Flatten: (28, 28) → (784,)  |  No learned parameters
    keras.layers.Flatten(input_shape=(28, 28)),

    #  Hidden Layer 1 
    # Dense: 784→50 neurons  |  W∈ℝ^{50×784}, b∈ℝ^{50}  |  ReLU activation
    keras.layers.Dense(50, activation='relu'),

    #  Hidden Layer 2 
    # Dense: 50→50 neurons   |  W∈ℝ^{50×50},  b∈ℝ^{50}  |  ReLU activation
    keras.layers.Dense(50, activation='relu'),

    #  Output Layer 
    # Dense: 50→10 neurons   |  W∈ℝ^{10×50},  b∈ℝ^{10}  |  Sigmoid per class
    keras.layers.Dense(10, activation='sigmoid')

], name='MNIST_FCNN')

# Display full parameter summary
model.summary()

---
<a id='compilation'></a>

## 7. Model Compilation & Optimization Strategy

### 7.1 Optimizer — Adam

We use **Adam** (*Adaptive Moment Estimation*, Kingma & Ba, 2015), the dominant first-order optimizer in modern deep learning. Adam adapts the learning rate per-parameter by tracking both first and second moment estimates of gradients:

$$
m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t \qquad \text{(1st moment — mean)}
$$
$$
v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \qquad \text{(2nd moment — uncentered variance)}
$$

Bias-corrected estimates and the update rule:
$$
\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}
$$
$$
\boldsymbol{\theta}_{t+1} = \boldsymbol{\theta}_t - \frac{\alpha}{\sqrt{\hat{v}_t} + \epsilon}\, \hat{m}_t
$$

| Hyperparameter | Default Value | Meaning |
|:---|:---|:---|
| $\alpha$ | $0.001$ | Global learning rate |
| $\beta_1$ | $0.9$ | Exponential decay for mean estimate |
| $\beta_2$ | $0.999$ | Exponential decay for variance estimate |
| $\epsilon$ | $10^{-8}$ | Numerical stability constant |

### 7.2 Loss Function — Sparse Categorical Cross-Entropy

For a batch of $N$ samples with integer ground-truth labels $\{y_i\}$ and predicted probability vectors $\{\hat{\mathbf{p}}_i\}$:

$$
\mathcal{L}(\boldsymbol{\theta}) = -\frac{1}{N} \sum_{i=1}^{N} \log \hat{p}_{i,\, y_i}
$$

This is the **negative log-likelihood** of the true class under the model's predicted distribution. The loss is minimized when the model assigns probability 1 to the correct class.

### 7.3 Evaluation Metric — Classification Accuracy

$$
\text{Accuracy} = \frac{1}{N} \sum_{i=1}^{N} \mathbb{1}\!\left[\hat{y}_i = y_i\right], \qquad \hat{y}_i = \arg\max_k \hat{p}_{i,k}
$$

Appropriate here given the near-balanced MNIST class distribution.

In [ ]:
# 
# § 7.1 — Compile: attach optimizer, loss function, and evaluation metric
# 
model.compile(
    optimizer='adam',                         # Adam with default lr=0.001
    loss='sparse_categorical_crossentropy',   # Accepts integer labels directly
    metrics=['accuracy']                      # Track classification accuracy per epoch
)

print("Model compiled ")
print("  Optimizer : Adam  (α=0.001, β₁=0.9, β₂=0.999)")
print("  Loss      : Sparse Categorical Cross-Entropy")
print("  Metric    : Accuracy")

---
<a id='training'></a>

## 8. Model Training & Learning Dynamics

### 8.1 The Training Loop

`model.fit()` initiates the iterative **forward-backward pass** cycle. For each mini-batch within each epoch:

$$
\underbrace{\hat{\mathbf{y}} = f_{\boldsymbol{\theta}}(\mathbf{X})}_{\textbf{1. Forward Pass}}
\xrightarrow{\quad}
\underbrace{\mathcal{L} = \text{CrossEntropy}(\hat{\mathbf{y}}, \mathbf{y})}_{\textbf{2. Loss Computation}}
\xrightarrow{\quad}
\underbrace{\nabla_{\boldsymbol{\theta}} \mathcal{L} = \text{Backprop}(\mathcal{L})}_{\textbf{3. Backward Pass}}
\xrightarrow{\quad}
\underbrace{\boldsymbol{\theta} \leftarrow \boldsymbol{\theta} - \eta \nabla_{\boldsymbol{\theta}} \mathcal{L}}_{\textbf{4. Weight Update}}
$$

**Backpropagation** applies the chain rule of calculus to efficiently compute $\nabla_{\boldsymbol{\theta}} \mathcal{L}$ through all layers in $O(\text{params})$ time — the algorithm that made deep learning tractable.

### 8.2 Training Configuration

| Hyperparameter | Value | Notes |
|:---|:---|:---|
| Epochs | 10 | Full passes over the training set |
| Batch Size | 32 (Keras default) | Samples per gradient update |
| Training samples | 60,000 | Full training partition |
| Validation split | 10% | 6,000 samples held back for monitoring |
| Steps per epoch | ≈ 1,688 | Gradient updates per epoch |

>  **Why `validation_split=0.1`?** Monitoring validation loss and accuracy during training is essential for detecting **overfitting** early — the point where training performance continues improving but generalization degrades. In this notebook we observe mild overfitting (~2% train-test gap), which is acceptable for a shallow MLP on MNIST.

In [ ]:
# 
# § 8.1 — Train the model
# X_train_scaled  : (60000, 28, 28) float64, normalized to [0,1]
# y_train         : (60000,) uint8, integer class labels
# validation_split: hold out 10% of training data for live monitoring
# 
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=10,
    validation_split=0.1,   # 6,000 samples reserved for validation
    verbose=1
)

In [ ]:
# 
# § 8.2 — Visualize training dynamics: loss & accuracy per epoch
# A healthy run shows: decreasing loss, increasing accuracy,
# with train and validation curves close together (low overfitting)
# 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history.history['loss']) + 1)

# Loss 
axes[0].plot(epochs_range, history.history['loss'],     'b-o', label='Training Loss',   linewidth=2, markersize=5)
axes[0].plot(epochs_range, history.history['val_loss'], 'r--s', label='Validation Loss', linewidth=2, markersize=5)
axes[0].set_title('Loss over Epochs', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Sparse Categorical Cross-Entropy', fontsize=11)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(list(epochs_range))

# Accuracy 
axes[1].plot(epochs_range, history.history['accuracy'],     'b-o', label='Training Accuracy',   linewidth=2, markersize=5)
axes[1].plot(epochs_range, history.history['val_accuracy'], 'r--s', label='Validation Accuracy', linewidth=2, markersize=5)
axes[1].set_title('Accuracy over Epochs', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(list(epochs_range))

plt.suptitle('Training Dynamics — MNIST FCNN', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
<a id='evaluation'></a>

## 9. Evaluation on Unseen Test Data

### 9.1 The Train/Test Performance Gap

Distinguishing **training performance** from **test performance** is foundational to the scientific validity of any ML experiment:

| Metric | Expected Value | What It Measures |
|:---|:---|:---|
| Training Accuracy | ≈ 99% | Fit to training distribution — includes memorization |
| Test Accuracy | ≈ 97% | **Generalization** — performance on novel unseen data |

The ~2% gap is the empirical **generalization error** — a direct consequence of mild overfitting. The model has captured genuine statistical regularities in the data (high test accuracy) but has also memorized some training-set-specific noise (gap > 0).

### 9.2 Bias-Variance Decomposition

The expected test error decomposes as:
$$
\mathbb{E}[(\hat{y} - y)^2] = \underbrace{\text{Bias}^2}_{\text{underfitting}} + \underbrace{\text{Variance}}_{\text{overfitting}} + \underbrace{\sigma^2_{\epsilon}}_{\text{irreducible noise}}
$$

Our model sits comfortably in the low-bias, low-variance regime for MNIST. On harder datasets (CIFAR-10, ImageNet), explicit variance reduction techniques become critical: **Dropout**, **L2 weight decay**, **early stopping**, **data augmentation**.

In [ ]:
# 
# § 9.1 — Evaluate generalization on the held-out test partition
# This is the FIRST and ONLY time the test set is used
# 
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

print("" * 40)
print(f"  Test Loss     : {test_loss:.4f}")
print(f"  Test Accuracy : {test_accuracy * 100:.2f} %")
print("" * 40)

---
<a id='inference'></a>

## 10. Probabilistic Inference & Prediction Decoding

### 10.1 `model.predict()` — Raw Output Tensor

`model.predict()` performs a vectorized **forward pass** over the entire input array and returns the raw output of the final Sigmoid layer — a matrix $\hat{\mathbf{P}} \in [0,1]^{N \times K}$:

$$
\hat{\mathbf{P}} = f_{\boldsymbol{\theta}}(\mathbf{X}_{\text{test}}) \in [0,1]^{10000 \times 10}
$$

Row $i$ of $\hat{\mathbf{P}}$ gives the model's **confidence scores** for all 10 digit classes for test image $i$.

### 10.2 Argmax Decoding

To convert continuous confidence scores into discrete class predictions:

$$
\hat{y}_i = \arg\max_{k \in \{0,\ldots,9\}} \hat{P}_{i,k}
$$

In NumPy: `np.argmax(Y_pred_probs, axis=1)` — returns the column index of the maximum value for each row.

In [ ]:
# 
# § 10.1 — Forward pass on all 10,000 test images
# Returns raw sigmoid output: shape (10000, 10)
# 
Y_pred_probs = model.predict(X_test_scaled)

print(f"Prediction matrix shape : {Y_pred_probs.shape}")
print(f"  Rows   = {Y_pred_probs.shape[0]:,} test samples")
print(f"  Cols   = {Y_pred_probs.shape[1]} class confidence scores")
print(f"  Range  : [{Y_pred_probs.min():.4f}, {Y_pred_probs.max():.4f}]")

In [ ]:
# 
# § 10.2 — Decode probability vectors to discrete class labels
# np.argmax(axis=1) → index of max value in each row
# 
Y_pred_labels = np.argmax(Y_pred_probs, axis=1)

print(f"Predicted labels : shape {Y_pred_labels.shape}  | dtype {Y_pred_labels.dtype}")
print()
print(f"First 20 predictions : {Y_pred_labels[:20]}")
print(f"First 20 true labels : {y_test[:20]}")
print()
correct = np.sum(Y_pred_labels == y_test)
print(f"Correct predictions  : {correct:,} / {len(y_test):,}  ({correct/len(y_test)*100:.2f}%)")

In [ ]:
# 
# § 10.3 — Detailed inspection: image + full probability distribution
# for a single test sample
# 
sample_idx = 29
probs = Y_pred_probs[sample_idx]
pred  = Y_pred_labels[sample_idx]
true  = y_test[sample_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: the image
axes[0].imshow(X_test[sample_idx], cmap='gray')
axes[0].set_title(f'Test Image [{sample_idx}]\nTrue Label: {true}', fontsize=12)
axes[0].axis('off')

# Right: class confidence bar chart
bar_colors = ['crimson' if i == pred else 'steelblue' for i in range(10)]
bars = axes[1].bar(range(10), probs, color=bar_colors, edgecolor='black', linewidth=0.7, alpha=0.85)
axes[1].set_xticks(range(10))
axes[1].set_xlabel('Digit Class', fontsize=12)
axes[1].set_ylabel('Sigmoid Confidence Score', fontsize=11)
axes[1].set_title(f'Class Probability Distribution\nPredicted: {pred}  |  True: {true}', fontsize=12)
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, axis='y', alpha=0.3)
axes[1].axhline(y=0.5, color='orange', linestyle='--', linewidth=1, label='Decision threshold')
axes[1].legend(fontsize=10)

plt.suptitle(f'Prediction Deep-Dive — Test Sample [{sample_idx}]', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Tabular output
print(f"{'Class':>7}  {'Confidence':>12}  {'':5}")
print("" * 35)
for i, p in enumerate(probs):
    marker = "  predicted" if i == pred else ""
    star   = "  true"      if i == true  else ""
    print(f"  Digit {i}  {p:.6f}  {marker}{star}")

---
<a id='confusion'></a>

## 11. Confusion Matrix Analysis

### 11.1 Definition

The **confusion matrix** $\mathbf{C} \in \mathbb{Z}^{K \times K}$ provides a complete per-class breakdown of classification performance:

$$
C_{ij} = \left| \left\{ n \,:\, y^{(n)} = i \text{ and } \hat{y}^{(n)} = j \right\} \right|
$$

- **Diagonal** $C_{ii}$: Correctly classified samples for class $i$ (true positives)
- **Off-diagonal** $C_{ij}$ ($i \neq j$): Samples of true class $i$ misclassified as $j$

A perfect classifier would produce a purely diagonal matrix.

### 11.2 Expected Failure Modes

Off-diagonal clusters in the MNIST confusion matrix typically reveal **visually similar digit pairs**:

| Confused Pair | Visual Reason |
|:---|:---|
| 4 ↔ 9 | Shared closed-loop upper region |
| 3 ↔ 8 | Similar curved structures |
| 5 ↔ 6 | Similar upper stroke patterns |
| 1 ↔ 7 | Both dominated by vertical strokes |

These confusions arise because our FCNN has **no spatial inductive bias** — it treats each pixel independently without reasoning about local structure. This is precisely what Convolutional layers solve.

In [ ]:
# 
# § 11.1 — Compute 10×10 confusion matrix
# 
conf_mat = confusion_matrix(y_test, Y_pred_labels)
print("10×10 Confusion Matrix (rows=True, cols=Predicted):")
print("" * 60)
print(conf_mat.numpy())

In [ ]:
# 
# § 11.2 — Annotated heatmap + per-class accuracy report
# 
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Heatmap 
sns.heatmap(
    conf_mat, annot=True, fmt='d', cmap='Blues',
    linewidths=0.5, linecolor='lightgray',
    xticklabels=range(10), yticklabels=range(10),
    ax=axes[0], cbar_kws={'label': 'Sample Count'}
)
axes[0].set_title('Confusion Matrix\n(MNIST Test Set — 10,000 samples)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].set_xlabel('Predicted Label', fontsize=12)

# Per-class accuracy bar chart 
conf_np    = conf_mat.numpy()
class_accs = [conf_np[i, i] / conf_np[i].sum() * 100 for i in range(10)]
bar_colors = ['#2ecc71' if a >= 97 else '#e67e22' if a >= 95 else '#e74c3c' for a in class_accs]

axes[1].bar(range(10), class_accs, color=bar_colors, edgecolor='black', linewidth=0.7)
axes[1].axhline(y=97, color='green',  linestyle='--', linewidth=1.5, label='97% threshold')
axes[1].axhline(y=95, color='orange', linestyle='--', linewidth=1.5, label='95% threshold')
axes[1].set_xticks(range(10))
axes[1].set_xlabel('Digit Class', fontsize=12)
axes[1].set_ylabel('Per-Class Accuracy (%)', fontsize=12)
axes[1].set_title('Per-Class Accuracy', fontsize=13, fontweight='bold')
axes[1].set_ylim(90, 100.5)
axes[1].legend(fontsize=11)
axes[1].grid(True, axis='y', alpha=0.3)
for i, acc in enumerate(class_accs):
    axes[1].text(i, acc + 0.2, f'{acc:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Model Performance Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("Per-Class Accuracy Report:")
print("" * 40)
for i, acc in enumerate(class_accs):
    status = '' if acc >= 97 else '' if acc >= 95 else ''
    print(f"  Digit {i}: {acc:6.2f}%  {status}")

---
<a id='predictive'></a>

## 12. End-to-End Predictive System

### 12.1 Inference Pipeline Design

Deploying a trained model requires a **preprocessing pipeline** that transforms arbitrary user inputs into the exact distribution seen during training. Any mismatch between training-time and inference-time preprocessing is a form of **distribution shift** that degrades performance.

For our MNIST model, the inference pipeline must:

```
  Raw Image File
       
       
  1. Read → cv2.imread(path, IMREAD_GRAYSCALE)     # Force single channel
       
       
  2. Resize → cv2.resize(img, (28, 28))            # Match training resolution
       
       
  3. Normalize → img / 255.0                       # Match training distribution
       
       
  4. Reshape → np.expand_dims(img, axis=0)         # Add batch dimension: (28,28)→(1,28,28)
       
       
  5. Forward Pass → model.predict(img_input)       # Shape: (1, 10)
       
       
  6. Decode → np.argmax(predictions)               # Scalar: 0–9
```

>  **Polarity Warning:** MNIST stores digits as **white strokes on black background**. If your custom image has black strokes on white background (typical pen-on-paper scan), you must **invert** it prior to inference:
> ```python
> img = cv2.bitwise_not(img)  # Invert: 0↔255
> ```

In [ ]:
# 
# § 12.1 — End-to-End Inference Pipeline
# 

def predict_digit(image_path: str, model, invert: bool = False, show: bool = True) -> dict:
    """
    End-to-end inference pipeline for single handwritten digit classification.

    Parameters
    ----------
    image_path : str
        Path to input image (JPEG, PNG, BMP, etc.)
    model : keras.Model
        Trained classification model (expects input shape (N, 28, 28))
    invert : bool, default False
        Set True if input has black digits on white background
        (inverts polarity to match MNIST convention)
    show : bool, default True
        Display the preprocessed image and probability distribution

    Returns
    -------
    dict with keys: 'predicted_digit', 'confidence', 'all_probs'
    """
    #  Step 1: Read image in grayscale 
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not load image: {image_path}")

    #  Step 2: Optional polarity inversion 
    if invert:
        img = cv2.bitwise_not(img)   # Flip: black-on-white → white-on-black

    #  Step 3: Resize to 28×28 (MNIST resolution) 
    img_resized = cv2.resize(img, (28, 28), interpolation=cv2.INTER_AREA)

    #  Step 4: Normalize to [0, 1] 
    img_normalized = img_resized / 255.0

    #  Step 5: Add batch dimension → (1, 28, 28) 
    img_input = np.expand_dims(img_normalized, axis=0)

    #  Step 6: Forward pass 
    probs           = model.predict(img_input, verbose=0)[0]   # Shape: (10,)
    predicted_digit = int(np.argmax(probs))
    confidence      = float(np.max(probs))

    #  Step 7: Display 
    if show:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        axes[0].imshow(img_resized, cmap='gray')
        axes[0].set_title(f'Input Image (28×28)', fontsize=12)
        axes[0].axis('off')

        colors = ['crimson' if i == predicted_digit else 'steelblue' for i in range(10)]
        axes[1].bar(range(10), probs, color=colors, edgecolor='black', linewidth=0.7)
        axes[1].set_xticks(range(10))
        axes[1].set_xlabel('Digit Class', fontsize=12)
        axes[1].set_ylabel('Confidence Score', fontsize=12)
        axes[1].set_title(f'Model Output  |  Predicted: {predicted_digit}  (conf: {confidence:.4f})',
                          fontsize=12, fontweight='bold')
        axes[1].set_ylim(0, 1.05)
        axes[1].grid(True, axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

    print("" * 48)
    print(f"  Image Path       : {image_path}")
    print(f"  Predicted Digit  : {predicted_digit}")
    print(f"  Model Confidence : {confidence:.4f}")
    print("" * 48)

    return {'predicted_digit': predicted_digit, 'confidence': confidence, 'all_probs': probs}


print("Predictive system ready ")
print("Usage: predict_digit('/path/to/digit.jpeg', model, invert=False)")

In [ ]:
# 
# § 12.2 — Upload a custom digit image and run inference
# Supported formats: JPEG, PNG, BMP
# Tip: Draw a digit on white paper, photograph it, and upload here
# 
from google.colab import files

print("Upload a handwritten digit image (JPG or PNG):")
uploaded = files.upload()

if uploaded:
    filename   = list(uploaded.keys())[0]
    image_path = f'/content/{filename}'
    print(f"\n'{filename}' uploaded   Running inference...\n")

    # Set invert=True if your image has black digit on white background
    result = predict_digit(image_path, model, invert=False, show=True)
else:
    print("No file uploaded.")

---
<a id='conclusion'></a>

## 13. Conclusion, Limitations & Future Directions

### 13.1 Results Summary

| Component | Details |
|:---|:---|
| **Architecture** | Flatten → Dense(50, ReLU) → Dense(50, ReLU) → Dense(10, Sigmoid) |
| **Total Parameters** | 42,810 |
| **Training Accuracy** | ≈ 99% |
| **Test Accuracy** | ≈ 97.06% |
| **Optimizer** | Adam (α=0.001) |
| **Loss** | Sparse Categorical Cross-Entropy |
| **Epochs** | 10 |

We have demonstrated that a **parameter-efficient, 2-hidden-layer feedforward network** trained for just 10 epochs achieves **97%+ test accuracy** on MNIST — establishing a strong and interpretable baseline for this series.

### 13.2 Key Lessons

1. **Normalization is non-negotiable.** Dividing by 255 had a direct, measurable impact on convergence speed.
2. **ReLU enables deeper training.** Its unit gradient for positive activations prevents the vanishing gradient problem that afflicts sigmoid/tanh in hidden layers.
3. **Adam converges reliably.** Per-parameter adaptive learning rates eliminate the need for careful manual learning rate tuning on standard benchmarks.
4. **Confusion analysis reveals structure.** The most common errors (4↔9, 3↔8) reflect genuine visual ambiguity — not model defects.

### 13.3 Known Limitations

| Limitation | Root Cause | Mitigation (Future Episodes) |
|:---|:---|:---|
| Spatial structure ignored | `Flatten` destroys pixel locality | **Convolutional Neural Networks (Episode 02)** |
| Mild overfitting (~2% gap) | No regularization | **Dropout, L2 decay, Batch Normalization** |
| Sigmoid output (not Softmax) | Independent class scores | **Proper probabilistic output** |
| Fixed architecture | No systematic search | **Keras Tuner, Optuna** |
| No data augmentation | Training distribution limited | **Random rotations, shifts, noise** |
| No model persistence | Model lost on kernel restart | **`model.save()` / `model.load_model()`** |

---
<a id='references'></a>

## References

1. **LeCun, Y., Cortes, C., & Burges, C.J.C.** (1998). *The MNIST Database of Handwritten Digits.* http://yann.lecun.com/exdb/mnist/
2. **Goodfellow, I., Bengio, Y., & Courville, A.** (2016). *Deep Learning.* MIT Press. https://www.deeplearningbook.org
3. **Kingma, D.P. & Ba, J.** (2015). *Adam: A Method for Stochastic Optimization.* ICLR 2015. arXiv:1412.6980
4. **Nair, V. & Hinton, G.E.** (2010). *Rectified Linear Units Improve Restricted Boltzmann Machines.* ICML 2010.
5. **Rumelhart, D.E., Hinton, G.E., & Williams, R.J.** (1986). *Learning Representations by Back-propagating Errors.* Nature 323, 533–536.
6. **Chollet, F.** (2021). *Deep Learning with Python, 2nd Edition.* Manning Publications.

---

<div align='center'>

**Deep Learning From Scratch — Episode 01 | MNIST Digit Classification**

*Next Episode → Episode 02: Convolutional Neural Networks (CNNs) for Image Recognition*

---

*If this notebook helped you, please  the repository and share it with fellow learners.*

</div>